## Langchain 라이브러리의 청킹(Chunking) 관련 클래스와 함수

- CharacterTextSplitter
  -  텍스트를 지정된 기준으로 나눠주는 기본적인 클래스
  -  텍스트가 간단한 구조일 때 적절(잘 정돈된 텍스트, csv 파일 등)
  - 구분자는 디폴트로 "\n\n"을 사용하지만 변경 가능
  - 구분자가 단일 문자이기 때문에, 연속된 텍스트의 의미를 충분히 고려하지 못할 수 있음

- RecursiveCharacterTextSplitter
  - 텍스트를 여러 단계를 거쳐 나누어주는 클래스(청킹에서 가장 많이 활용됨)
  - 텍스트가 상대적으로 복잡한 구조일 때 적절(일반적인 문서, 웹페이지, 보고서 등)
  - 리스트를 통해 구분자를 여러 개 사용하여 세분화 된 청크로 만들 수 있음(구분자의 순서가 중요함)
  - 처음엔 "\n\n"으로 분리하고 청크가 여전히 크다면 "\n", 그래도 크다면 " " 이런식으로 지정된 크기 제한 내에 들어올 때까지 계층적 분할을 진행
  - 계층적 분할을 통해 의미가 있는 단위가 가능한 온전하게 유지되도록 함

**간단한 텍스트에서는 CharacterTextSplitter가 Recursive에 비해 더 속도가 빨라 대용량 처리에 유리할 수 있음**

In [ ]:
%pip install -U langchain-text-splitters

In [2]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

In [3]:
text1 = '청킹을 위한 예제 데이터입니다. 어떻게 쪼개지는지 알아볼까요?'
text2 = '청킹을 위한 예제 데이터입니다. 어떻게 쪼개지는지 알아볼까요? 결과를 확인해봅시다.'
text3 = '''청킹을 위한 예제 데이터 입니다.
어떻게 쪼개지는지 알아볼까요?

결과를 확인해봅시다. '''

### 1) Character Splitter

In [4]:
# 청킹 객체 생성

chunk_size = 20      # 하나의 청크에 대한 길이 기준을 문자 20개로 설정
chunk_overlap = 0

c_splitter = CharacterTextSplitter(
    # 청크 크기 지정
    chunk_size = chunk_size,
    # 청크 간 겹치는 부분 크기 지정
    chunk_overlap = chunk_overlap
    )

In [5]:
# split_text : 텍스트 데이터 청킹 실행 함수
c_splitter.split_text(text1)

['청킹을 위한 예제 데이터입니다. 어떻게 쪼개지는지 알아볼까요?']

In [6]:
c_splitter.split_text(text2)

['청킹을 위한 예제 데이터입니다. 어떻게 쪼개지는지 알아볼까요? 결과를 확인해봅시다.']

In [7]:
c_splitter.split_text(text3)

# 무조건 chunk_size보다 작은 크기의 청크로 분리되는 것은 아니고, 의미 있는 문장이라면 splitter의 조건에 따라 더 길게 출력될 수 있음

Created a chunk of size 35, which is longer than the specified 20


['청킹을 위한 예제 데이터 입니다.\n어떻게 쪼개지는지 알아볼까요?', '결과를 확인해봅시다.']

- 발생한 경고 메세지는 지정된 20자 크기 기준보다 더 큰 청크가 생성되었다는 뜻으로 CharacterTextSplitter 의 디폴트인 '\n\n' 이 텍스트 내에서 충분히 자주 발생하지 않아서 20자 이내로 나눌 수 없는 경우 발생됨

### 2) Recursive Splitter

In [ ]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap
    )

In [ ]:
r_splitter.split_text(text1)

In [ ]:
r_splitter.split_text(text2)

In [ ]:
r_splitter.split_text(text3)

### Character, Recursive 비교

In [ ]:
see_you_again = """It's been a long day without you, my friend.
And I'll tell you all about it when I see you again.
We've come a long way from where we began.
Oh, I'll tell you all about it when I see you again.
When I see you again.


Damn, who knew?
All the planes we flew, good things we been through.
That I'd be standing right here talking to you.
'Bout another path, I know we loved to hit the road and laugh.
But something told me that it wouldn't last.
Had to switch up, look at things different, see the bigger picture.
Those were the days, hard work forever pays.
Now I see you in a better place (see you in a better place).
Uh."""

In [ ]:
len(see_you_again)

In [ ]:
# 너 없는 긴 하루였어, 내 친구
# 그리고 너를 다시 만나면 그것에 대해 모두 말해 줄게
# 우리는 우리가 시작한 곳에서 먼 길을 왔고
# 오, 너를 다시 만나면 그것에 대해 모두 말해 줄게
# 널 다시 만나면
#
# 젠장, 누가 알았겠어?
# 우리가 날아간 모든 비행기들, 우리가 겪은 좋은 일들
# 내가 바로 여기 서서 당신과 이야기를 하고 있다는 것,
# 다른 길에 대해, 우리가 길을 가며 웃는 것을 좋아했다는 것을 알아요
# 하지만 어떤 것이 그것이 지속되지 않을 것이라고 말했어요
# 일어나, 다른 것을 봐, 더 큰 그림을 봐
# 그 시절은 좋았어, 노력은 영원히 보상받을 거야
# 이제 더 나은 곳에서 만나 (더 나은 곳에서 만나)
# Uh

In [ ]:
# separator로 청크로 나누는 기준을 설정할 수 있음
c_splitter = CharacterTextSplitter(chunk_size=100,
                                   chunk_overlap=0,
                                   separator = '\n\n'    # 직접 지정 가능
                                  )
c_splitter.split_text(see_you_again)

In [ ]:
r_splitter = RecursiveCharacterTextSplitter(chunk_size=100,
                                            chunk_overlap=0,
                                            separators=["\n\n", "\n", " "]   # 리스트로 계층적 조건을 설정 가능
                                           )
r_splitter.split_text(see_you_again)

- CharacterTextSplitter를 사용한 결과가 만족스럽지 못하다면 더 작은 단위의 구분자를 지정("\n", " ")하여 해결하거나 RecursiveCharacterTextSplitter를 사용하여 계층적 분할을 적용하면 됨
- 즉 CharacterTextSplitter보다 RecursiveCharacterTextSplitter가 더욱 사용하기 편리하고 결과가 좋음

## 웹 페이지에 접근해 쉽게 텍스트 데이터 가져오기

#### LangChain(2022.10)
- 자연어 처리(NLP)와 사전 학습된 LLM모델을 사용하여 다양한 응용프로그램을 구축할 수 있게 해주는 라이브러리(LangSmith로 설계 및 배포기능까지 지원)
- 데이터베이스, API, 외부 서비스(ChatGPT 등)와의 통합을 지원하는 인터페이스 제공(openai 외에도 Anthropic, Hugging Face, Google 등 다양한 기업들의 LLM 모델들과 연동)
- RAG 구현을 위한 기본적인 인터페이스와 임베딩, Advanced RAG 등의 기능을 제공
- 공식 페이지 : https://www.langchain.com/
- langchain docs 페이지 : https://python.langchain.com/v0.2/docs/introduction/

In [ ]:
# langchain 설치
!pip install -U langchain

In [ ]:
# langchain과 openai의 GPT모델을 결합한 기능을 이용
!pip install -U langchain-openai

In [ ]:
# langchain 생태계를 확장하고 기본 langchain기능을 보완하는 다양한 도구와의 통합을 제공
!pip install -U langchain_community

In [ ]:
# WebBaseLoader : langchain라이브러리에서 지원하는 웹 페이지 HTML 콘텐츠를 로드해주는 클래스
#                 (내부적으로 BeautifulSoup을 사용함)
from langchain.document_loaders import WebBaseLoader

# WARNING은 웹 접근시 user_agent 설정이 필요하다는 경고 문구

In [ ]:
loader = WebBaseLoader("https://n.news.naver.com/mnews/article/003/0012627594")

# SSL 인증 오류 우회 설정
# loader.requests_kwargs = {'verify': False}

# 브라우저 우회 접속
loader.requests_kwargs = {'headers': {'User-Agent': 'Mozilla/5.0'}}   # 개발자도구 - Network - 요소 클릭후 우측 Headers탭 확인
# 한글 인코딩 설정
loader.encoding = 'utf-8'

docs = loader.load()

In [ ]:
docs[0].page_content[:500]

In [ ]:
print(docs[0].page_content[:500])

- 문자열 처리 코드나 정규표현식 등을 활용하여 필요한 텍스트만 추출해서 사용할 수 있음

### 텍스트 정제
#### 1) 문자열 함수 replace 활용

In [ ]:
docs[0].page_content[:500].replace("\n", "")

#### 2) 문자열 정규표현식 활용
- 정규표현식은 검색 결과가 개별 문자열로 출력되기 때문에 필요에 따라 이를 모두 합쳐서 텍스트로 활용해야 함

In [ ]:
import re

In [ ]:
# 문자열 앞 r 의 의미는 문자열 내에 이스케이프 코드를 지칭하는 \(역슬래시)가 있을 경우 이를 일반 문자열로 인지하게 함
# 가-힣 : 한글 문자열 전체
# a-zA-Z : 영문 문자열(대,소문자 포함) 전체
# 0-9_ : 숫자 및 언더바(_) 전체
# + : 연속된 문자열 반환(띄워쓰기 기준)
re.findall(r"[\w]+", docs[0].page_content[:500])   # [\w]+ 작성시 리터럴 경고가 발생하여 앞에 r 붙여줄 것

In [ ]:
" ".join(re.findall(r"[\w]+", docs[0].page_content))